In [21]:
import json

with open("tim_data.json") as fp:
    data = json.load(fp)
    plays = data["plays"]

In [22]:
import numpy as np
from pandas import DataFrame, read_json

df = DataFrame(plays).replace(np.nan, None)

In [23]:
df.columns

Index(['uuid', 'modificationDate', 'entryDate', 'playDate', 'usesTeams',
       'durationMin', 'ignored', 'manualWinner', 'rounds', 'bggId',
       'bggLastSync', 'importPlayId', 'gameRefId', 'comments', 'rating',
       'nemestatsId', 'scoringSetting', 'metaData', 'playerScores',
       'expansionPlays', 'locationRefId', 'scoresheet', 'board', 'playImages'],
      dtype='object')

In [24]:
game_names = {game["id"]: game["name"] for game in data["games"]}
df["Game Name"] = df["gameRefId"].map(game_names)
df = df.drop(columns="gameRefId")

In [25]:
df = df.drop(
    columns=[
        "uuid",
        "playImages",
        "metaData",
        "nemestatsId",
        "scoresheet",
        "importPlayId",
    ]
)
df = df.rename(
    columns={
        "uuid": "Play ID",
        "modificationDate": "Last Modified",
        "entryDate": "Entry Date",
        "playDate": "Play Date",
        "usesTeams": "Uses Teams",
        "durationMin": "Duration Minutes",
        "ignored": "Ignored",
        "manualWinner": "Manual Winner",
        "rounds": "Round Count",
        "bggId": "BGG Game ID",
        "bggLastSync": "BGG Last Sync",
        "importPlayId": "Import Play ID",
        "comments": "Comments",
        "rating": "Rating",
        "nemestatsId": "Nemestats ID",
        "scoringSetting": "Scoring Setting",
        "metaData": "Metadata",
        "playerScores": "Player Scores",
        "expansionPlays": "Expansion Plays",
        "locationRefId": "Location Ref ID",
        "scoresheet": "Scoresheet",
        "board": "Board",
        "playImages": "Play Images",
    }
)

In [26]:
from datetime import datetime


df["Play Date"] = df["Play Date"].apply(lambda strtime: datetime.strptime(strtime, "%Y-%m-%d %H:%M:%S"))



In [27]:
df_copy = df.copy()
years = set(dt.year for dt in df["Play Date"])
for year in years:
    df_copy[str(year)] = df["Play Date"].apply(lambda dt: int(dt.year == year))
numeric_columns = [c for c in df_copy.select_dtypes(include="number").columns if "20" in c]
grouped_game = df_copy.groupby(by="Game Name")[numeric_columns].sum()
del df_copy

In [28]:
grouped_game["Years"] = grouped_game.apply(lambda row: [year for year in range(2000, 2026) if row.get(str(year), 0) > 0], axis=1)

In [29]:
def consecutive_years(years: list[int]) -> int:
    last_number, res, count = 0, 0, 0
    for year in years:
        if last_number + 1 == year:
            count += 1
        else:
            count = 1
        last_number = year
        res = max(res, count)
    return res

def latest_years(years: list[int]) -> int:
    for i in range(25):
        if 2025-i not in years:
            return i
    return 0

def salvageable_streak(years: list[int]) -> int | None:
    if 2025 in years or 2024 not in years:
        return None
    for i in range(25):
        if 2024-i not in years:
            return i
    return None

grouped_game["Longest Consecutive Years"] = grouped_game["Years"].apply(consecutive_years)
grouped_game["Consecutive Years Streak"] = grouped_game["Years"].apply(latest_years)
grouped_game["Salvageable Streak"] = grouped_game["Years"].apply(salvageable_streak)

In [30]:
from pandas import ExcelWriter


with ExcelWriter("tim.xlsx", engine="openpyxl") as xlsx:
    grouped_game.to_excel(xlsx, sheet_name="Sum of Columns")
    df.to_excel(xlsx, sheet_name="Plays")
    for sheet in data:
        if sheet == "plays":
            continue
        try:
            DataFrame(data[sheet]).to_excel(xlsx, sheet_name=sheet)
        except:
            print(sheet)

userInfo
